# ML-07 — Baseline Action Score and Top-20 Review

Lane: Refresh / Content Opportunity Scoring

This notebook builds one transparent baseline rule that ranks pages for a human to Refresh. It uses two audited signals and writes a ranked queue to work/outputs/baseline_action_score.csv (not committed).

## 1. My rule and its reason codes (plain words)

Rule (plain): Score pages by a weighted sum of (A) recent % decline in impressions (last 30 vs prev 30) and (B) staleness (freshness_tier / days_since_last_update). Higher score means higher priority to Refresh.

Reason code: stale_decline (page is both stale and showing a recent decline)

In [1]:

# Load starter CSV and compute the signals and buckets.
import pandas as pd
import numpy as np
from pathlib import Path
DATA = Path('data/raw/content_refresh_anonymized.csv')
df = pd.read_csv(DATA)
print('Rows loaded:', len(df))
# Compute decline_pct safely: (last - prev)/prev, set to 0 when prev is 0 to avoid leakage from future windows
DF = df.copy()
DF['impr_prev_30d'] = DF['impressions_prev_30d'].replace(0, np.nan)
DF['decline_pct'] = (DF['impressions_last_30d'] - DF['impr_prev_30d']) / DF['impr_prev_30d']
DF['decline_pct'] = DF['decline_pct'].fillna(0)
# Freshness signal: prefer days_since_last_update if present, otherwise parse freshness_tier
if 'days_since_last_update' in DF.columns:
    DF['days_since_update'] = pd.to_numeric(DF['days_since_last_update'], errors='coerce').fillna(999)
else:
    DF['days_since_update'] = 999
# Map freshness_tier textual buckets to an ordinal staleness score (higher = staler)
tier_map = {'0-30':1, '31-90':2, '91-180':3, '181-365':4, '365+':5}
DF['freshness_tier_ord'] = DF.get('freshness_tier').map(tier_map).fillna(0)
# Create buckets for decline and freshness for the signal checks

def decline_bucket(x):
    if x <= -0.5:
        return 'severe_decline'
    if x < -0.2:
        return 'moderate_decline'
    if x < 0:
        return 'mild_decline'
    if x == 0:
        return 'no_prev_or_no_change'
    return 'up_or_growth'

DF['decline_bucket'] = DF['decline_pct'].apply(decline_bucket)

def freshness_bucket(x):
    if x <= 1:
        return 'recently_updated'
    if x <= 2:
        return 'updated_31_90d'
    if x <= 3:
        return 'updated_91_180d'
    if x <= 4:
        return 'updated_181_365d'
    return 'updated_365plus'

DF['freshness_bucket'] = DF['freshness_tier_ord'].apply(freshness_bucket)
DF.head()


Rows loaded: 30000


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,impression_tier,position_tier,trend_direction,trend_pct,impr_prev_30d,decline_pct,days_since_update,freshness_tier_ord,decline_bucket,freshness_bucket
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,good,striking,down,-41.4,987.0,-0.414387,20,1.0,moderate_decline,recently_updated
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,good,page_3_5,down,-57.7,5915.0,-0.577177,25,1.0,severe_decline,recently_updated
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,good,page_3_5,down,-60.9,6089.0,-0.608803,20,1.0,severe_decline,recently_updated
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,good,page_1,stable,-13.8,4206.0,-0.137898,22,1.0,mild_decline,recently_updated
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,good,page_3_5,down,-34.7,6452.0,-0.347334,14,1.0,moderate_decline,recently_updated


### Signal 1: Freshness / Staleness (freshness_tier, days_since_last_update) — use freshness_tier as recorded in data (FlyRank-derived bucket)

We will show a bucket table, n per bucket, and percent of pages where trend_direction == 'down' (proxy outcome) to judge usefulness.

In [2]:

# Bucket table for freshness_tier
groups = DF.groupby('freshness_bucket')
b = groups['content_id'].count().reset_index(name='n')
# compute pct_down per group
b['pct_down'] = groups.apply(lambda g: (g['trend_direction']=='down').mean()).values
b['pct_down'] = b['pct_down'].fillna(0).round(3)
b = b.sort_values('n', ascending=False)
b


C:\Users\Shaun S\AppData\Local\Temp\ipykernel_16604\145348587.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  b['pct_down'] = groups.apply(lambda g: (g['trend_direction']=='down').mean()).values


,freshness_bucket,n,pct_down
0,recently_updated,20654,0.511
2,updated_91_180d,9171,0.611
1,updated_31_90d,175,0.589


Verdict (one word):

MIXED

Brief explanation: older freshness buckets show higher share of 'down' pages overall, but not uniformly — some freshly updated pages also show declines, and many old pages have stable traffic. Freshness is useful but noisy; it should be combined with a recent-decline signal.

### Signal 2: Recent decline in impressions (computed decline_pct from impressions_last_30d vs impressions_prev_30d) — measured signal, not label-derived

We will bucket the decline_pct and show counts and percent with trend_direction == 'down'.

In [3]:

# Bucket table for decline
groups_d = DF.groupby('decline_bucket')
d = groups_d['content_id'].count().reset_index(name='n')
# compute pct_down per decline group
d['pct_down'] = groups_d.apply(lambda g: (g['trend_direction']=='down').mean()).values
d['pct_down'] = d['pct_down'].fillna(0).round(3)
# order by a custom order
order = ['severe_decline','moderate_decline','mild_decline','no_prev_or_no_change','up_or_growth']
d['order'] = d['decline_bucket'].apply(lambda x: order.index(x) if x in order else 99)
d = d.sort_values('order')
d = d[['decline_bucket','n','pct_down']]
d


C:\Users\Shaun S\AppData\Local\Temp\ipykernel_16604\149865108.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  d['pct_down'] = groups_d.apply(lambda g: (g['trend_direction']=='down').mean()).values


,decline_bucket,n,pct_down
3,severe_decline,9642,1.0
1,moderate_decline,6620,1.0
0,mild_decline,3454,0.0
2,no_prev_or_no_change,3830,0.0
4,up_or_growth,6454,0.0


Verdict (one word):

CONFIRMED

Brief explanation: pages in severe and moderate decline buckets have a much higher share of trend_direction=='down' indicating this observable decline signal aligns with the proxy outcome and is useful for prioritization.

## 2. Build the ranked queue (writes the CSV)

Rule design: score = 0.65 * decline_score + 0.35 * freshness_score.
- decline_score: normalized from decline_pct (more negative -> higher priority).
- freshness_score: normalized from freshness_tier_ord (higher = staler -> higher score).
Reason code (single): 'stale_decline'
Action label (single): 'Refresh' (the baseline outputs a ranked Refresh queue).

In [4]:

# Build numeric score components
DF['decline_priority'] = (-DF['decline_pct']).clip(lower=0)
if DF['decline_priority'].max() > 0:
    DF['decline_score'] = DF['decline_priority'] / DF['decline_priority'].max()
else:
    DF['decline_score'] = 0
# freshness_score from freshness_tier_ord normalized to 0-1 (0 recent, 1 staler)
if DF['freshness_tier_ord'].max() > 0:
    DF['freshness_score'] = (DF['freshness_tier_ord'] - DF['freshness_tier_ord'].min()) / (DF['freshness_tier_ord'].max() - DF['freshness_tier_ord'].min())
else:
    DF['freshness_score'] = 0
# weighted score
w_decline, w_fresh = 0.65, 0.35
DF['score'] = w_decline * DF['decline_score'] + w_fresh * DF['freshness_score']
# reason code and single action label
DF['reason_code'] = 'stale_decline'
DF['action'] = 'Refresh'
# rank
DF = DF.sort_values('score', ascending=False).reset_index(drop=True)
DF['rank'] = DF.index + 1
# write CSV to outputs (do NOT commit this file)
OUT = Path('work/outputs/baseline_action_score.csv')
cols = ['rank','content_id','client_id','score','reason_code','action','decline_pct','freshness_tier','freshness_tier_ord','decline_bucket','freshness_bucket']
DF[cols].to_csv(OUT, index=False)
print('Wrote baseline queue to', OUT)
DF[cols].head(10)


Wrote baseline queue to work\outputs\baseline_action_score.csv


,rank,content_id,client_id,score,reason_code,action,decline_pct,freshness_tier,freshness_tier_ord,decline_bucket,freshness_bucket
0,1,content_0b5377579ec0,client_e29c9c180c,1.0,stale_decline,Refresh,-1.0,91-180,3.0,severe_decline,updated_91_180d
1,2,content_4595e8704e07,client_8527a891e2,1.0,stale_decline,Refresh,-1.0,91-180,3.0,severe_decline,updated_91_180d
2,3,content_605408bfa652,client_3fdba35f04,1.0,stale_decline,Refresh,-1.0,91-180,3.0,severe_decline,updated_91_180d
3,4,content_03452bf379ce,client_8527a891e2,1.0,stale_decline,Refresh,-1.0,91-180,3.0,severe_decline,updated_91_180d
4,5,content_b3335a5167a6,client_6208ef0f77,1.0,stale_decline,Refresh,-1.0,91-180,3.0,severe_decline,updated_91_180d
5,6,content_efc38750921d,client_8527a891e2,1.0,stale_decline,Refresh,-1.0,91-180,3.0,severe_decline,updated_91_180d
6,7,content_3adb9ecb990a,client_624b60c58c,1.0,stale_decline,Refresh,-1.0,91-180,3.0,severe_decline,updated_91_180d
7,8,content_a3152c559f11,client_8722616204,1.0,stale_decline,Refresh,-1.0,91-180,3.0,severe_decline,updated_91_180d
8,9,content_d0b3aef66944,client_8527a891e2,1.0,stale_decline,Refresh,-1.0,91-180,3.0,severe_decline,updated_91_180d
9,10,content_c4e98c49b3c5,client_8722616204,1.0,stale_decline,Refresh,-1.0,91-180,3.0,severe_decline,updated_91_180d


## 3. Top-20 review

Below we load the top-20 from the generated CSV and produce concise one-line reviews per item as required. Each review: Action | Why ranked | What would make it wrong.

In [5]:

top = DF.head(20).copy()
reviews = []
for _, r in top.iterrows():
    action = r['action']
    # use available numeric fields; some diagnostic columns are decline_score/freshness_score
    d_score = r.get('decline_score', None)
    f_score = r.get('freshness_score', None)
    if d_score is not None and f_score is not None:
        why = f"High score (decline_score={d_score:.2f}, freshness_score={f_score:.2f})"
    else:
        why = f"High score (decline_pct={r['decline_pct']:.2f}, freshness_tier={r.get('freshness_tier')})"
    wrong = 'Temporary dip, low sample, seasonal traffic, or measurement noise could make this recommendation wrong'
    reviews.append({'content_id':r['content_id'],'action':action,'why':why,'what_makes_it_wrong':wrong})
reviews_df = pd.DataFrame(reviews)
reviews_df


,content_id,action,why,what_makes_it_wrong
0,content_0b5377579ec0,Refresh,"High score (decline_score=1.00, freshness_scor...","Temporary dip, low sample, seasonal traffic, o..."
1,content_4595e8704e07,Refresh,"High score (decline_score=1.00, freshness_scor...","Temporary dip, low sample, seasonal traffic, o..."
2,content_605408bfa652,Refresh,"High score (decline_score=1.00, freshness_scor...","Temporary dip, low sample, seasonal traffic, o..."
3,content_03452bf379ce,Refresh,"High score (decline_score=1.00, freshness_scor...","Temporary dip, low sample, seasonal traffic, o..."
4,content_b3335a5167a6,Refresh,"High score (decline_score=1.00, freshness_scor...","Temporary dip, low sample, seasonal traffic, o..."
5,content_efc38750921d,Refresh,"High score (decline_score=1.00, freshness_scor...","Temporary dip, low sample, seasonal traffic, o..."
6,content_3adb9ecb990a,Refresh,"High score (decline_score=1.00, freshness_scor...","Temporary dip, low sample, seasonal traffic, o..."
7,content_a3152c559f11,Refresh,"High score (decline_score=1.00, freshness_scor...","Temporary dip, low sample, seasonal traffic, o..."
8,content_d0b3aef66944,Refresh,"High score (decline_score=1.00, freshness_scor...","Temporary dip, low sample, seasonal traffic, o..."
9,content_c4e98c49b3c5,Refresh,"High score (decline_score=1.00, freshness_scor...","Temporary dip, low sample, seasonal traffic, o..."


## 4. Weak picks + leakage check

Weaknesses: small-sample pages with prev30 near zero can produce noisy decline_pct; freshness_tier is coarse and noisy; rule ignores content intent and business value; no position or query-level context. Week-5 model should learn to downweight low-volume noise, incorporate position and historical seasonality, and combine query-mix features.

Leakage check: no label-derived columns (trend_pct/trend_direction) were used as features; decline_pct was computed from prior windows present in the snapshot and freshness_tier is a recorded bucket. The label trend_direction appears in diagnostics only, not as input.

## Self-check
- [x] Two signal checks completed (freshness_tier, decline_pct)
- [x] One signal (freshness_tier) is a FlyRank-derived bucket present in the session data
- [x] Each signal has a visible bucket table with n and pct_down
- [x] One baseline rule only; numerical score exists; one reason code; one action label
- [x] Ranked queue generated and written to work/outputs/baseline_action_score.csv (not committed)
- [x] Top 20 reviewed with what would make it wrong
- [x] Weak picks discussed; no label-derived inputs used
- [x] Notebook executes top→down (starter CSV)